# 05 — Interpretable Machine Learning (IML)

This notebook interprets the predictions of our final bike-availability model (`models/lightgbmv6.joblib`) with three complementary techniques, from coarse to fine:

1. **PDP / ICE** — Partial Dependence & Individual Conditional Expectation
2. **ALE** — Accumulated Local Effects (robust to correlated features)
3. **Shapley values** — a from-scratch Monte-Carlo estimate, then the **SHAP** library

It is self-contained: it loads the saved model and rebuilds the feature matrix exactly as `04_prediction.ipynb` does, with no new `src/` code.

> **Two things to keep in mind**
> - We interpret on the **test set** (the rows we actually submit). There is **no ground-truth target** here, so every plot describes what the *model predicts*, not what truly happens.
> - The model is a `TransformedTargetRegressor(log1p / expm1)` around LightGBM, so the inner tree ensemble reasons in **log1p(bikes)** space. PDP/ALE below call the full pipeline (counts), while SHAP explains the inner model (log space) — we flag this where it matters.

## 0 · Environment setup

Same local preamble as notebooks 03/04: put the project root on `sys.path` and run from it so `src/` imports and relative data paths resolve.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# IML dependencies were added to requirements.txt: matplotlib, shap, PyALE
!pip install -r requirements.txt -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
import shap
from PyALE import ale
from sklearn.inspection import PartialDependenceDisplay

RANDOM_STATE = 42
print("shap", shap.__version__)

## 1 · Load the model and rebuild the feature matrix

Same recipe as `04_prediction.ipynb`: clean the raw test set, drop the identifier columns (`id`, `datetime`, `name`), and feed the rest straight to the model. `station_number` stays a pandas `category` — LightGBM consumes it natively.

In [ ]:
from src.data_cleaning import clean_data

# Raw test set -> cleaned features (identical to notebook 04)
df_test = pd.read_csv("data/raw/dataset_test.csv")
df_test = clean_data(dataset=df_test, is_train=False, categorize_station=True)

DROP_COLS = ["id", "datetime", "name"]
X_full = df_test.drop(columns=DROP_COLS)
print(X_full.shape)
X_full.dtypes

In [ ]:
model = joblib.load("models/lightgbmv6.joblib")
print(model)

# IML methods scale with rows x (grid points or permutations); a representative
# subsample keeps every section interactive while preserving the feature
# distribution. Bump N_SAMPLE up for sharper plots if you have time to wait.
N_SAMPLE = 20_000
X = X_full.sample(n=min(N_SAMPLE, len(X_full)), random_state=RANDOM_STATE).sort_index()

# Predicted bikes (count space) for the sampled rows — a reference throughout.
y_pred = np.maximum(0, model.predict(X))
print(f"Explaining {len(X):,} rows | mean predicted bikes = {y_pred.mean():.2f}")

## 2 · Partial Dependence (PDP) & ICE

**PDP** shows the *average* predicted bikes as one feature is swept across its range, marginalising over all other features. **ICE** draws one line per row, exposing the heterogeneity that the average hides.

**Caveat — feature independence.** PDP forces a feature to grid values while holding the rest at their real combinations, so for **correlated** features — here `hour` vs `hour_sin`/`hour_cos`, and `avg_bikes_per_station` vs `station_number` — it evaluates the model on combinations that never occur. ALE (next section) fixes exactly this.

In [ ]:
# sklearn PDP (and PyALE) write float grid values back into the feature column
# in place. On recent pandas that raises LossySetitemError / "Invalid value ...
# for dtype 'int64'" for integer columns (hour, month, relative_humidity_2m, ...).
# Cast the integer features to float for the PDP/ALE views — LightGBM scores them
# identically; station_number stays a category. SHAP below uses the original X.
int_cols = X.select_dtypes("integer").columns
X_pdp = X.astype({c: "float64" for c in int_cols})

pdp_features = ["hour", "temperature_2m", "avg_bikes_per_station",
                "wind_speed_10m", "month", "relative_humidity_2m"]

fig, ax = plt.subplots(figsize=(14, 8))
PartialDependenceDisplay.from_estimator(
    model, X_pdp, features=pdp_features,
    kind="average", grid_resolution=30, n_jobs=-1, ax=ax,
)
fig.suptitle("Partial Dependence — average predicted bikes", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ICE (individual lines) for the strongest temporal driver, plus a 2-way PDP
# that shows how the hour effect bends with temperature. Uses the float-cast
# X_pdp from the previous cell (see the note there).
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

PartialDependenceDisplay.from_estimator(
    model, X_pdp, features=["hour"], kind="both",
    subsample=200, grid_resolution=30, random_state=RANDOM_STATE, ax=ax[0],
)
ax[0].set_title("ICE + PDP — hour")

PartialDependenceDisplay.from_estimator(
    model, X_pdp, features=[("hour", "temperature_2m")],
    grid_resolution=20, ax=ax[1],
)
ax[1].set_title("2-way PDP — hour x temperature")
fig.tight_layout()
plt.show()

## 3 · Accumulated Local Effects (ALE)

ALE asks a *local* question: within small bins of a feature, how much does the prediction change when the feature moves a little? It then accumulates those local changes across bins. Because it perturbs each feature only **within bins of the real data**, it stays on the data manifold and is **unbiased under feature correlation** — unlike PDP.

Compare the ALE curves for `hour` and `avg_bikes_per_station` against their PDPs above: where they disagree, correlation was distorting the PDP.

In [ ]:
# PyALE perturbs a single feature within data bins and calls model.predict
# (count space). It draws its own figure per feature (with a CI band). Uses the
# float-cast X_pdp so integer columns don't trip the same in-place assignment.
ale_features = ["hour", "temperature_2m", "avg_bikes_per_station", "wind_speed_10m"]

for feat in ale_features:
    ale(X=X_pdp, model=model, feature=[feat], grid_size=30, include_CI=True, C=0.95)
    plt.title(f"ALE — {feat}")
    plt.show()

## 4 · Shapley values from scratch

Think of a single prediction as a *payout* and each feature as a *player*. The Shapley value fairly splits the gap between this row's prediction and the average prediction across the features, by averaging each feature's **marginal contribution** over all the orderings in which it could be added to a coalition.

Exact Shapley needs all $2^{19}$ feature coalitions — infeasible. We approximate it with **Monte-Carlo permutation sampling**: draw random feature orderings, swap features one at a time from a background row into the explained row, and average the marginal prediction changes. We work in the inner model's **log1p space** (`model.regressor_`) so the result lines up with SHAP's `TreeExplainer` in the next section.

*Efficiency property:* the contributions should sum to `f(x) - E[f]`, i.e. `sum(phi) + base_value` should match the model's log-prediction for the row.

In [ ]:
# Pick a random observation and compute ITS Shapley values from scratch, via
# Monte-Carlo permutation sampling in the inner model's log1p space
# (model.regressor_) so they line up with SHAP's TreeExplainer below.
rng = np.random.default_rng(RANDOM_STATE)
predict_log = model.regressor_.predict          # inner LightGBM -> log1p(bikes)

row_idx = int(rng.integers(len(X)))             # the randomly chosen observation
instance = X.iloc[[row_idx]]
X_background = X.sample(n=200, random_state=RANDOM_STATE)   # reference distribution
features = list(X.columns)
n_perm = 100                                    # raise for a tighter estimate

phi = {f: 0.0 for f in features}
for _ in range(n_perm):
    # Start from a random background row (copy keeps the category dtypes intact)
    z = X_background.iloc[[rng.integers(len(X_background))]].copy()
    base = predict_log(z)[0]
    for f in rng.permutation(features):
        z[f] = instance[f].values               # reveal this feature's true value
        nxt = predict_log(z)[0]
        phi[f] += nxt - base                    # marginal contribution
        base = nxt

phi = pd.Series({f: v / n_perm for f, v in phi.items()}).sort_values()
base_value = predict_log(X_background).mean()

print(f"Random observation: row {row_idx} | station {instance['station_number'].iat[0]} | "
      f"hour {int(instance['hour'].iat[0])} | predicted bikes = {model.predict(instance)[0]:.2f}")
print(f"sum(phi) + base_value = {phi.sum() + base_value:.4f}   "
      f"(model log-prediction = {predict_log(instance)[0]:.4f})\n")
print("Shapley values (log1p space), most positive last:")
print(phi)

phi.plot.barh(figsize=(8, 7))
plt.title(f"Shapley values (log1p space) — random row {row_idx}")
plt.xlabel("contribution to log1p(bikes)")
plt.tight_layout()
plt.show()

## 5 · SHAP (TreeExplainer)

SHAP computes the same Shapley values, but for tree ensembles `TreeExplainer` does it **exactly and fast** via the path-dependent algorithm — no sampling. We explain the inner `model.regressor_`, so all SHAP values and the base value live in **log1p(bikes)** space, consistent with the manual estimate above.

In [ ]:
explainer = shap.TreeExplainer(model.regressor_)
shap_exp = explainer(X)            # Explanation object, log1p space

# Global importance — beeswarm shows direction + magnitude per feature
shap.plots.beeswarm(shap_exp, max_display=15, show=True)

In [ ]:
# Mean |SHAP| ranking, and a dependence plot for the hour effect coloured
# by temperature (the same interaction we probed with the 2-way PDP).
shap.plots.bar(shap_exp, max_display=15, show=True)
shap.plots.scatter(shap_exp[:, "hour"], color=shap_exp[:, "temperature_2m"], show=True)

In [ ]:
# Local explanation for the same random observation we did by hand (row_idx)
shap.plots.waterfall(shap_exp[row_idx], max_display=15, show=True)

# Cross-check: manual Monte-Carlo Shapley vs exact TreeExplainer for that row.
# They should agree closely (small gaps are Monte-Carlo noise).
compare = pd.DataFrame({
    "manual_MC": phi,
    "shap_tree": pd.Series(shap_exp[row_idx].values, index=X.columns),
})
compare["abs_diff"] = (compare["manual_MC"] - compare["shap_tree"]).abs()
print(f"manual base_value = {base_value:.4f} | shap base_value = {float(shap_exp[row_idx].base_values):.4f}")
compare.sort_values("shap_tree").tail(12)

## 6 · Wrap-up

- **PDP/ICE** gave a first, intuitive read of each feature's average effect (clear daily `hour` cycle, monotone lift from `avg_bikes_per_station`), but assumes feature independence.
- **ALE** re-checked those effects without that assumption; divergences from the PDP flag where correlation (`hour` ↔ `hour_sin/cos`, `avg_bikes_per_station` ↔ `station_number`) was biasing the PDP.
- **Shapley / SHAP** moved from global to local: the manual Monte-Carlo estimate showed *how* the values are built, and `TreeExplainer` delivered them exactly — global beeswarm/bar rankings plus per-row waterfalls — with the two agreeing on row 0.

Together they describe the model consistently: `avg_bikes_per_station` and time-of-day dominate, weather plays a secondary modulating role. Remember the SHAP/manual values are in **log1p(bikes)** space, while PDP/ALE are in raw counts.